# # Business Data Lab - Business Conditions Survey Data Analysis

 This notebook analyzes four quarterly Business Conditions Survey files:

 - Q3 2023
 - Q4 2023
 - Q1 2024
 - Q2 2024

 The analysis focuses on:

 1. National business expectations over time using a Net Expectation measure.
 2. Provincial employment expectations for Q2 2024.

A series of data-quality checks are also performed before analysis.

In [ ]:
import pandas as pd
import altair as alt
from pathlib import Path

print("Packages installed successfully")

Packages installed successfully


# Import Data

Four quarterly CSV files were provided. Each file follows the same general
structure but contains data for a different survey quarter.

In [81]:
q3_2023 = pd.read_csv("Data_CSBC-Q3_2023.csv")
q4_2023 = pd.read_csv("Data_CSBC-Q4_2023.csv")
q1_2024 = pd.read_csv("Data_CSBC-Q1_2024.csv")
q2_2024 = pd.read_csv("Data_CSBC-Q2_2024.csv")

print("Q3 2023 shape:", q3_2023.shape)
print("Q4 2023 shape:", q4_2023.shape)
print("Q1 2024 shape:", q1_2024.shape)
print("Q2 2024 shape:", q2_2024.shape)

Q3 2023 shape: (2352, 6)
Q4 2023 shape: (2352, 6)
Q1 2024 shape: (2352, 6)
Q2 2024 shape: (2352, 6)


# Inspect Columns
Before combining the files, verify that the quarterly files have the
expected fields.


In [ ]:
for quarter, data in {
    "Q3 2023": q3_2023,
    "Q4 2023": q4_2023,
    "Q1 2024": q1_2024,
    "Q2 2024": q2_2024
}.items():

    print(f"\n{quarter} columns:")
    print(data.columns.tolist())


Q3 2023 columns:
['GEO', 'Business_characteristics', 'Business_information', 'Expected_change', 'VALUE', 'Quarter']

Q4 2023 columns:
['GEO', 'Business_characteristics', 'Business_information', 'Expected_change', 'VALUE', 'Quarter']

Q1 2024 columns:
['GEO', 'Business_characteristics', 'Business_information', 'Expected_change', 'VALUE', 'Quarter']

Q2 2024 columns:
['GEO', 'Business_characteristics', 'Business_information', 'Expected_change', 'VALUE', 'Quarter']


# Check Missing Values in Source Files

Missing values are inspected before transformation so that potential
data-quality issues can be distinguished from issues introduced during
processing.



In [ ]:
quarterly_data = {
    "Q3 2023": q3_2023,
    "Q4 2023": q4_2023,
    "Q1 2024": q1_2024,
    "Q2 2024": q2_2024
}

for quarter, data in quarterly_data.items():

    print(f"\n{quarter}")
    display(data.isna().sum())


Q3 2023


GEO                         0
Business_characteristics    0
Business_information        0
Expected_change             0
VALUE                       0
Quarter                     0
dtype: int64


Q4 2023


GEO                         0
Business_characteristics    0
Business_information        0
Expected_change             0
VALUE                       0
Quarter                     0
dtype: int64


Q1 2024


GEO                         0
Business_characteristics    0
Business_information        0
Expected_change             0
VALUE                       0
Quarter                     0
dtype: int64


Q2 2024


GEO                         0
Business_characteristics    0
Business_information        0
Expected_change             0
VALUE                       7
Quarter                     0
dtype: int64

# Combine Quarterly Data

The quarterly files are appended into one dataset so that trends over time can be analyzed consistently.

In [ ]:
df = pd.concat(
    [
        q3_2023,
        q4_2023,
        q1_2024,
        q2_2024
    ],
    ignore_index=True
)

print("Combined dataset shape:", df.shape)

display(df.head())

Combined dataset shape: (9408, 6)


,GEO,Business_characteristics,Business_information,Expected_change,VALUE,Quarter
0,Canada,North American Industry Classification System ...,Employment,increase,11.6,Q3 2023
1,Canada,North American Industry Classification System ...,Employment,stay about the same,79.6,Q3 2023
2,Canada,North American Industry Classification System ...,Employment,decrease,8.8,Q3 2023
3,Canada,North American Industry Classification System ...,Sales,increase,18.4,Q3 2023
4,Canada,North American Industry Classification System ...,Sales,stay about the same,62.4,Q3 2023


# Verify Quarter Counts

In [ ]:
print(
    df["Quarter"].value_counts()
)

Quarter
Q3 2023    2352
Q4 2023    2352
Q1 2024    2352
Q2 2024    2352
Name: count, dtype: int64


# Clean and Standardize Data

Column names are standardized to lowercase with underscores. \
Text values are stripped of unnecessary whitespace. \
Known category-label differences between quarterly files are standardized.

In [ ]:
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Strip whitespace from text columns
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    df[col] = df[col].str.strip()

# Standardize expected-change labels
df["expected_change"] = (
    df["expected_change"]
    .str.lower()
    .replace({
        "stay about the same": "stay the same"
    })
)

# Standardize business-information labels
df["business_information"] = (
    df["business_information"]
    .replace({
        "Capital Investment": "Investment"
    })
)

# Convert VALUE to numeric
df["value"] = pd.to_numeric(
    df["value"],
    errors="coerce"
)

print("Standardized columns:")
print(df.columns.tolist())

print("\nExpected-change categories:")
print(df["expected_change"].unique())

print("\nBusiness-information categories:")
print(df["business_information"].unique())

Standardized columns:
['geo', 'business_characteristics', 'business_information', 'expected_change', 'value', 'quarter']

Expected-change categories:
<ArrowStringArray>
['increase', 'stay the same', 'decrease']
Length: 3, dtype: str

Business-information categories:
<ArrowStringArray>
['Employment', 'Sales', 'Profitability', 'Investment']
Length: 4, dtype: str


# Data Quality Checks
## Missing Values

In [ ]:
print("Missing values by column:")

display(
    df.isna().sum()
)

Missing values by column:


geo                         0
business_characteristics    0
business_information        0
expected_change             0
value                       7
quarter                     0
dtype: int64

# VALUE Range Check
VALUE represents a percentage, so valid observations should fall between 0 and 100.

In [ ]:
invalid_values = df[
    (df["value"] < 0) |
    (df["value"] > 100)
]

print(
    "Number of values outside 0-100:",
    len(invalid_values)
)

if len(invalid_values) > 0:
    display(invalid_values)

Number of values outside 0-100: 0


# Expected-Change Category Check

In [ ]:
expected_categories = {
    "increase",
    "stay the same",
    "decrease"
}

actual_categories = set(
    df["expected_change"].dropna().unique()
)

print("Expected categories:")
print(expected_categories)

print("\nObserved categories:")
print(actual_categories)

print("\nUnexpected categories:")
print(actual_categories - expected_categories)

Expected categories:
{'stay the same', 'increase', 'decrease'}

Observed categories:
{'stay the same', 'increase', 'decrease'}

Unexpected categories:
set()


## Three-Way Percentage Split Check

For each geography, business characteristic, business-information category, and quarter, the **increase**, **stay the same**, and **decrease** percentages should sum to approximately 100%.

A tolerance of one percentage point is used to account for rounding.

In [ ]:
split_group_cols = [
    "geo",
    "business_characteristics",
    "business_information",
    "quarter"
]

split_check = (
    df
    .groupby(split_group_cols)["value"]
    .sum()
    .reset_index(name="total")
)

split_check["difference_from_100"] = (
    split_check["total"] - 100
)

problem_groups = split_check[
    split_check["difference_from_100"].abs() > 1
]

print(
    "Total groups checked:",
    len(split_check)
)

print(
    "Groups more than 1 percentage point from 100:",
    len(problem_groups)
)

display(problem_groups.head(20))

Total groups checked: 3136
Groups more than 1 percentage point from 100: 1396


,geo,business_characteristics,business_information,quarter,total,difference_from_100
4,Alberta,Accommodation and food services,Investment,Q1 2024,96.0,-4.0
5,Alberta,Accommodation and food services,Investment,Q2 2024,91.9,-8.1
6,Alberta,Accommodation and food services,Investment,Q3 2023,92.7,-7.3
7,Alberta,Accommodation and food services,Investment,Q4 2023,89.7,-10.3
9,Alberta,Accommodation and food services,Profitability,Q2 2024,98.7,-1.3
20,Alberta,"Agriculture, forestry, fishing and hunting",Investment,Q1 2024,98.9,-1.1
21,Alberta,"Agriculture, forestry, fishing and hunting",Investment,Q2 2024,93.6,-6.4
22,Alberta,"Agriculture, forestry, fishing and hunting",Investment,Q3 2023,93.4,-6.6
23,Alberta,"Agriculture, forestry, fishing and hunting",Investment,Q4 2023,95.2,-4.8
36,Alberta,"Arts, entertainment and recreation",Investment,Q1 2024,91.2,-8.8


# National All-Industry Dataset

The first analysis focuses on businesses across Canada using the **"North American Industry Classification System (NAICS), all industries"** grouping.

In [ ]:
all_industries = (
    "North American Industry Classification System (NAICS), all industries"
)

national_all_industry = df[
    (df["geo"] == "Canada") &
    (df["business_characteristics"] == all_industries)
].copy()

print(
    "National all-industry observations:",
    len(national_all_industry)
)

display(
    national_all_industry.head()
)

National all-industry observations: 48


,geo,business_characteristics,business_information,expected_change,value,quarter
0,Canada,North American Industry Classification System ...,Employment,increase,11.6,Q3 2023
1,Canada,North American Industry Classification System ...,Employment,stay the same,79.6,Q3 2023
2,Canada,North American Industry Classification System ...,Employment,decrease,8.8,Q3 2023
3,Canada,North American Industry Classification System ...,Sales,increase,18.4,Q3 2023
4,Canada,North American Industry Classification System ...,Sales,stay the same,62.4,Q3 2023


# Validate National Three-Way Split
Verify that each quarter and business-information category contains the expected three response categories.

In [ ]:
national_category_counts = (
    national_all_industry
    .groupby(
        ["quarter", "business_information"]
    )["expected_change"]
    .nunique()
    .reset_index(
        name="number_of_categories"
    )
)

display(national_category_counts)

,quarter,business_information,number_of_categories
0,Q1 2024,Employment,3
1,Q1 2024,Investment,3
2,Q1 2024,Profitability,3
3,Q1 2024,Sales,3
4,Q2 2024,Employment,3
5,Q2 2024,Investment,3
6,Q2 2024,Profitability,3
7,Q2 2024,Sales,3
8,Q3 2023,Employment,3
9,Q3 2023,Investment,3


In [ ]:
display(
    national_all_industry[
        [
            "quarter",
            "business_information",
            "expected_change",
            "value"
        ]
    ]
    .sort_values(
        [
            "quarter",
            "business_information",
            "expected_change"
        ]
    )
)

,quarter,business_information,expected_change,value
4706,Q1 2024,Employment,decrease,6.0
4704,Q1 2024,Employment,increase,11.1
4705,Q1 2024,Employment,stay the same,82.9
4715,Q1 2024,Investment,decrease,9.3
4713,Q1 2024,Investment,increase,17.6
4714,Q1 2024,Investment,stay the same,57.0
4712,Q1 2024,Profitability,decrease,31.8
4710,Q1 2024,Profitability,increase,11.2
4711,Q1 2024,Profitability,stay the same,54.4
4709,Q1 2024,Sales,decrease,15.3


# Q2 2024 Missing Values

The Q2 2024 national all-industry observations contain missing values for the "decrease" category across the four business-information measures. These values are retained as missing rather than being replaced with zero, because zero would imply that no businesses expected a decrease.

In [ ]:
q2_decrease = national_all_industry[
    (national_all_industry["quarter"] == "Q2 2024") &
    (national_all_industry["expected_change"] == "decrease")
].copy()

print(
    "Number of Q2 decrease rows:",
    len(q2_decrease)
)

print(
    "Missing Q2 decrease values:",
    q2_decrease["value"].isna().sum()
)

display(
    q2_decrease[
        [
            "business_information",
            "expected_change",
            "value"
        ]
    ]
)

Number of Q2 decrease rows: 4
Missing Q2 decrease values: 4


,business_information,expected_change,value
7058,Employment,decrease,NaN
7061,Sales,decrease,NaN
7064,Profitability,decrease,NaN
7067,Investment,decrease,NaN


# Analysis 1: Net Expectation

Net expectation measures the balance between businesses expecting improvement and businesses expecting deterioration.

**Net Expectation = % expecting an increase − % expecting a decrease**

A positive value indicates that more businesses expect an increase than a decrease, while a negative value indicates the opposite.

Q2 2024 national net expectations are unavailable because the source data contains missing `decrease` percentages for all four business-information categories.

In [ ]:
net_expectation = (
    national_all_industry
    .pivot(
        index=[
            "quarter",
            "business_information"
        ],
        columns="expected_change",
        values="value"
    )
    .reset_index()
)

net_expectation.columns.name = None

# Calculate net expectation only where both components are available
net_expectation["net_expectation"] = (
    net_expectation["increase"]
    - net_expectation["decrease"]
)

display(net_expectation)

,quarter,business_information,decrease,increase,stay the same,net_expectation
0,Q1 2024,Employment,6.0,11.1,82.9,5.1
1,Q1 2024,Investment,9.3,17.6,57.0,8.3
2,Q1 2024,Profitability,31.8,11.2,54.4,-20.6
3,Q1 2024,Sales,15.3,17.8,63.1,2.5
4,Q2 2024,Employment,NaN,13.2,81.5,NaN
5,Q2 2024,Investment,NaN,17.6,56.3,NaN
6,Q2 2024,Profitability,NaN,13.3,53.7,NaN
7,Q2 2024,Sales,NaN,20.7,63.7,NaN
8,Q3 2023,Employment,8.8,11.6,79.6,2.8
9,Q3 2023,Investment,9.0,19.4,56.3,10.4


In [ ]:
display(
    net_expectation[
        [
            "quarter",
            "business_information",
            "increase",
            "decrease",
            "net_expectation"
        ]
    ]
)

,quarter,business_information,increase,decrease,net_expectation
0,Q1 2024,Employment,11.1,6.0,5.1
1,Q1 2024,Investment,17.6,9.3,8.3
2,Q1 2024,Profitability,11.2,31.8,-20.6
3,Q1 2024,Sales,17.8,15.3,2.5
4,Q2 2024,Employment,13.2,NaN,NaN
5,Q2 2024,Investment,17.6,NaN,NaN
6,Q2 2024,Profitability,13.3,NaN,NaN
7,Q2 2024,Sales,20.7,NaN,NaN
8,Q3 2023,Employment,11.6,8.8,2.8
9,Q3 2023,Investment,19.4,9.0,10.4


# Net Business Expectations Over Time

In [89]:
net_chart = (

    alt.Chart(net_expectation)

    .mark_line(point=True)

    .encode(

        x=alt.X(
            "quarter:N",
            sort=[
                "Q3 2023",
                "Q4 2023",
                "Q1 2024",
                "Q2 2024"
            ],
            title="Quarter"
        ),

        y=alt.Y(
            "net_expectation:Q",
            title="Net Expectation (percentage points)"
        ),

        color=alt.Color(
            "business_information:N",
            title="Business metric"
        ),

        tooltip=[
            alt.Tooltip(
                "quarter:N",
                title="Quarter"
            ),

            alt.Tooltip(
                "business_information:N",
                title="Business metric"
            ),

            alt.Tooltip(
                "net_expectation:Q",
                title="Net expectation",
                format=".1f"
            )
        ]

    )

    .properties(
        title="Net Business Expectations Over Time",
        width=700,
        height=400
    )
)

net_chart

alt.Chart(...)

### Interpretation

Net expectation provides a simple measure of whether business expectations are tilted toward improvement or deterioration. Positive values indicate that a larger share of businesses expect an increase than a decrease, while negative values indicate the opposite. The chart allows changes in expectations across Employment, Sales, Profitability, and Investment to be compared over time. Q2 2024 is not included in the net-expectation comparison because the source data contains missing "decrease" percentages for all four national all-industry measures; these values were not treated as zero because doing so would introduce an unsupported assumption. Overall, the net expectations of the 4 business metrics (employment, investment, profitability and sales) decreased in Q4 of 2023, but rebounded in Q1 of 2024. 

# Analysis 2: Provincial Employment Expectations in Q2 2024
The second analysis compares employment expectations across Canadian
provinces in Q2 2024.

In [ ]:
q2_employment = df[
    (df["quarter"] == "Q2 2024") &
    (df["business_information"] == "Employment") &
    (
        df["business_characteristics"]
        == all_industries
    )
].copy()

display(
    q2_employment
)

,geo,business_characteristics,business_information,expected_change,value,quarter
7056,Canada,North American Industry Classification System ...,Employment,increase,13.2,Q2 2024
7057,Canada,North American Industry Classification System ...,Employment,stay the same,81.5,Q2 2024
7058,Canada,North American Industry Classification System ...,Employment,decrease,NaN,Q2 2024
7224,Newfoundland and Labrador,North American Industry Classification System ...,Employment,increase,10.6,Q2 2024
7225,Newfoundland and Labrador,North American Industry Classification System ...,Employment,stay the same,85.3,Q2 2024
7226,Newfoundland and Labrador,North American Industry Classification System ...,Employment,decrease,4.1,Q2 2024
7392,Prince Edward Island,North American Industry Classification System ...,Employment,increase,20.2,Q2 2024
7393,Prince Edward Island,North American Industry Classification System ...,Employment,stay the same,75.2,Q2 2024
7394,Prince Edward Island,North American Industry Classification System ...,Employment,decrease,4.6,Q2 2024
7560,Nova Scotia,North American Industry Classification System ...,Employment,increase,19.6,Q2 2024


In [ ]:
provincial_employment = q2_employment[
    q2_employment["geo"] != "Canada"
].copy()

display(
    provincial_employment
)

,geo,business_characteristics,business_information,expected_change,value,quarter
7224,Newfoundland and Labrador,North American Industry Classification System ...,Employment,increase,10.6,Q2 2024
7225,Newfoundland and Labrador,North American Industry Classification System ...,Employment,stay the same,85.3,Q2 2024
7226,Newfoundland and Labrador,North American Industry Classification System ...,Employment,decrease,4.1,Q2 2024
7392,Prince Edward Island,North American Industry Classification System ...,Employment,increase,20.2,Q2 2024
7393,Prince Edward Island,North American Industry Classification System ...,Employment,stay the same,75.2,Q2 2024
7394,Prince Edward Island,North American Industry Classification System ...,Employment,decrease,4.6,Q2 2024
7560,Nova Scotia,North American Industry Classification System ...,Employment,increase,19.6,Q2 2024
7561,Nova Scotia,North American Industry Classification System ...,Employment,stay the same,76.2,Q2 2024
7562,Nova Scotia,North American Industry Classification System ...,Employment,decrease,4.2,Q2 2024
7728,New Brunswick,North American Industry Classification System ...,Employment,increase,18.1,Q2 2024


In [ ]:
provincial_employment_summary = (
    provincial_employment
    .pivot(
        index="geo",
        columns="expected_change",
        values="value"
    )
    .reset_index()
)

provincial_employment_summary.columns.name = None

provincial_employment_summary["net_expectation"] = (
    provincial_employment_summary["increase"]
    - provincial_employment_summary["decrease"]
)

display(
    provincial_employment_summary
)

,geo,decrease,increase,stay the same,net_expectation
0,Alberta,5.1,13.0,81.9,7.9
1,British Columbia,4.6,12.9,82.5,8.3
2,Manitoba,5.7,12.8,81.5,7.1
3,New Brunswick,3.7,18.1,78.2,14.4
4,Newfoundland and Labrador,4.1,10.6,85.3,6.5
5,Northwest Territories,9.7,28.3,62.0,18.6
6,Nova Scotia,4.2,19.6,76.2,15.4
7,Nunavut,5.2,15.2,79.6,10.0
8,Ontario,4.9,12.0,83.2,7.1
9,Prince Edward Island,4.6,20.2,75.2,15.6


## Provincial and Territorial Employment Expectations

In [90]:
employment_chart = (

    alt.Chart(provincial_employment_summary)

    .mark_bar()

    .encode(

        x=alt.X(
            "net_expectation:Q",
            title="Net Employment Expectation (percentage %)"
        ),

        y=alt.Y(
            "geo:N",
            sort="-x",
            title="Province"
        ),

        tooltip=[

            alt.Tooltip(
                "geo:N",
                title="Province"
            ),

            alt.Tooltip(
                "increase:Q",
                title="Expect increase",
                format=".1f"
            ),

            alt.Tooltip(
                "stay the same:Q",
                title="Expect same",
                format=".1f"
            ),

            alt.Tooltip(
                "decrease:Q",
                title="Expect decrease",
                format=".1f"
            ),

            alt.Tooltip(
                "net_expectation:Q",
                title="Net expectation",
                format=".1f"
            )

        ]

    )

    .properties(
        title="Provincial & Territorial Employment Expectations: Q2 2024",
        width=700,
        height=500
    )

    .interactive()
)

employment_chart

alt.Chart(...)

### Interpretation

Employment expectations vary across provinces, with the net expectation showing the balance between businesses expecting employment to increase and those expecting it to decrease. Provinces with higher positive net expectations have a stronger balance of businesses anticipating employment growth. This comparison is useful for identifying geographic differences in near-term business hiring expectations. NWT had the strongest net employment expectation at 28.3 percentage points, followed by PEI at 20.2, while Newfoundland and Labrador had the weakest at 10.6. The results should be interpreted as business expectations rather than forecasts of actual employment changes.

In [ ]:
# Provincial Data-Quality Check
provincial_category_counts = (
    provincial_employment
    .groupby("geo")["expected_change"]
    .nunique()
    .reset_index(
        name="number_of_categories"
    )
)

display(
    provincial_category_counts
)

,geo,number_of_categories
0,Alberta,3
1,British Columbia,3
2,Manitoba,3
3,New Brunswick,3
4,Newfoundland and Labrador,3
5,Northwest Territories,3
6,Nova Scotia,3
7,Nunavut,3
8,Ontario,3
9,Prince Edward Island,3


In [85]:
provincial_split_check = (
    provincial_employment
    .groupby("geo")["value"]
    .sum()
    .reset_index(name="total")
)

provincial_split_check["difference_from_100"] = (
    provincial_split_check["total"] - 100
)

display(
    provincial_split_check
)

,geo,total,difference_from_100
0,Alberta,100.0,0.000000e+00
1,British Columbia,100.0,0.000000e+00
2,Manitoba,100.0,0.000000e+00
3,New Brunswick,100.0,0.000000e+00
4,Newfoundland and Labrador,100.0,-1.421085e-14
5,Northwest Territories,100.0,0.000000e+00
6,Nova Scotia,100.0,0.000000e+00
7,Nunavut,100.0,0.000000e+00
8,Ontario,100.1,1.000000e-01
9,Prince Edward Island,100.0,0.000000e+00


# Conclusions and Assumptions

## Key findings

1. Net expectation provides a concise measure of the balance between businesses expecting an increase and those expecting a decrease. It allows changes in expectations for Employment, Sales, Profitability, and Investment to be compared consistently across available quarters.

2. Provincial employment expectations in Q2 2024 show geographic variation in the balance between businesses expecting employment increases and decreases. This provides a useful snapshot of where employment expectations are relatively more or less positive.

## Data-quality findings

The quarterly files were checked for missing values, invalid percentage values, unexpected category labels, and consistency of the three-way increase/stay-the-same/decrease split.

A notable data-quality issue was identified in the Q2 2024 national all-industry data: the "decrease" values are missing for Employment, Sales, Profitability, and Investment. These values were not replaced with zero because a missing value does not imply that zero businesses expected a decrease. Consequently, Q2 2024 national net expectations are reported as unavailable rather than being calculated using an unsupported assumption.

## Assumptions

- `VALUE` is interpreted as a percentage from 0 to 100.
- The increase, stay the same, and decrease categories are expected to represent the complete response distribution for each group.
- A tolerance of one percentage point is used when checking whether the three categories sum to 100 because survey percentages may be affected by rounding.
- Net expectation is calculated as the percentage expecting an increase minus the percentage expecting a decrease.
- Missing values are retained as missing rather than imputed.
- The national analysis uses Canada and the "North American Industry Classification System (NAICS), all industries" grouping.
- The provincial analysis excludes the Canada aggregate and compares individual provinces.